# 01 – Daten und Definitionen

Erst ein Blick in die Daten, dann die Entscheidungen, auf denen das Modell aufbaut:

1. **Die Daten im Überblick:** Event-Daten, Kader, Tracking-Daten
2. **Pass-Tabelle** aller 7 Spiele
3. **Was zählt als Pass?**
4. **Wann ist ein Pass erfolgreich?**
5. **Empfänger bei Fehlpässen**
6. **Leakage:** Welche Angaben dürfen ins Modell?

Die Pass-Tabelle baut `passdaten.py` im Projektordner. Am Ende speichern wir sie als `data/paesse.csv`; die weiteren Notebooks lesen sie von dort.

Beim ersten Ausführen lädt kloppy die Daten aus dem Internet und speichert sie in `~/kloppy_cache`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from kloppy import sportec

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from passdaten import MATCH_IDS, lade_events, pass_tabelle

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Die Daten im Überblick

Als Beispiel nehmen wir ein Spiel aus der 2. Bundesliga: Fortuna Düsseldorf – SSV Jahn Regensburg (`J03WOH`).

### 1.1 Event-Daten

Jede Aktion mit Ball ist ein Event: Pass, Zweikampf, Schuss, Foul … kloppy lädt die Events und macht mit `to_df()` eine Tabelle daraus, eine Zeile pro Event. Koordinaten in Metern, Ursprung im Mittelpunkt (x von −52,5 bis 52,5, y von −34 bis 34).

In [2]:
BEISPIEL = "J03WOH"
events = lade_events(BEISPIEL)
heim, gast = events.metadata.teams
print(f"{heim.name} – {gast.name}: {len(events.records)} Events")

ev = events.to_df()
ev[["event_type", "period_id", "timestamp", "team_id", "player_id", "coordinates_x", "coordinates_y",
    "end_coordinates_x", "end_coordinates_y", "result"]].head(10)

Fortuna Düsseldorf – SSV Jahn Regensburg: 1446 Events


,event_type,period_id,timestamp,team_id,player_id,coordinates_x,coordinates_y,end_coordinates_x,end_coordinates_y,result
0,PASS,1,0 days 00:00:00.000000,DFL-CLU-000011,DFL-OBJ-002GAB,0.00,0.00,-17.23,0.10,COMPLETE
1,PASS,1,0 days 00:00:05.702000,DFL-CLU-000011,DFL-OBJ-00268T,-17.23,0.10,19.93,28.51,COMPLETE
2,GENERIC:TacklingGame,1,0 days 00:00:06.719000,NaN,NaN,19.93,28.51,NaN,NaN,NaN
3,PASS,1,0 days 00:00:09.430000,DFL-CLU-000011,DFL-OBJ-002G2Y,21.24,31.09,44.38,31.69,COMPLETE
4,GENERIC:OtherBallAction,1,0 days 00:00:10.182000,DFL-CLU-000011,DFL-OBJ-0027XP,44.38,31.69,NaN,NaN,NaN
5,GENERIC:TacklingGame,1,0 days 00:00:10.830000,NaN,NaN,44.21,32.70,NaN,NaN,NaN
6,FOUL_COMMITTED,1,0 days 00:00:12.822000,DFL-CLU-00000P,DFL-OBJ-00006V,NaN,NaN,NaN,NaN,NaN
7,PASS,1,0 days 00:01:19.295000,DFL-CLU-000011,DFL-OBJ-0027B0,44.21,29.91,NaN,NaN,COMPLETE
8,CARD,1,0 days 00:00:16.290000,DFL-CLU-00000P,DFL-OBJ-00006V,NaN,NaN,NaN,NaN,NaN
9,GENERIC:Delete,1,0 days 00:01:19.856000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Welche Events gibt es, und wie oft?
ev["event_type"].value_counts().head(8)

event_type
PASS                       764
GENERIC:OtherBallAction    228
GENERIC:TacklingGame       198
BALL_OUT                    59
GENERIC:Delete              52
RECOVERY                    48
FOUL_COMMITTED              31
SHOT                        27
Name: count, dtype: int64

Über die Hälfte aller Events sind Pässe. Bei Pässen gibt `result` an, ob der Pass ankam (`COMPLETE`), beim Gegner landete (`INCOMPLETE`) oder ins Aus ging (`OUT`). Einen Endpunkt (`end_coordinates`) gibt es nur bei angekommenen Pässen.

**Rohdaten der DFL:** kloppy übernimmt nicht alle Angaben der DFL. Die vollständigen Rohdaten stehen in `raw_event`. So sieht das für einen Pass aus:

In [4]:
erster_pass = next(e for e in events.records if e.event_type.name == "PASS")
pd.Series(erster_pass.raw_event).to_frame("Wert")

,Wert
FreeKickLayup,false
Recipient,DFL-OBJ-00268T
Height,flat
Player,DFL-OBJ-002GAB
BallPossessionPhase,6
Evaluation,successfullyCompleted
PlayAngle,180.33
FromOpenPlay,false
SemiField,false
Distance,medium


Wichtig für uns: `Evaluation` (Bewertung des Passes), `Height` (flach/hoch), `PlayAngle` (Richtung in Grad, 0° = Richtung gegnerisches Tor), `Distance` (kurz/mittel/lang), `Direction` (Passart, z. B. Steilpass, nur manchmal ausgefüllt) und `Recipient` (Empfänger).

### 1.2 Kader

Die Metadaten enthalten Teams und Spieler. Damit übersetzen wir die Spieler-IDs in Namen.

In [5]:
kader = pd.DataFrame([
    dict(team=t.name, name=p.name, nummer=p.jersey_no, position=str(p.starting_position), startelf=p.starting)
    for t in events.metadata.teams for p in t.players
])
print(kader.groupby("team")["startelf"].agg(spieler="size", startelf="sum"))
kader.head(10)

                     spieler  startelf
team                                  
Fortuna Düsseldorf        20        11
SSV Jahn Regensburg       19        11


,team,name,nummer,position,startelf
0,Fortuna Düsseldorf,R. Hennings,28,Striker,False
1,Fortuna Düsseldorf,R. Wolf,1,Unknown,False
2,Fortuna Düsseldorf,D. Ginczek,10,Striker,True
3,Fortuna Düsseldorf,Matthias Zimmermann,25,Right Back,True
4,Fortuna Düsseldorf,A. Hoffmann,3,Right Center Back,True
5,Fortuna Düsseldorf,F. Klaus,11,Right Midfield,True
6,Fortuna Düsseldorf,Jorrit Hendrix,29,Left Defensive Midfield,True
7,Fortuna Düsseldorf,M. Sobottka,31,Right Defensive Midfield,True
8,Fortuna Düsseldorf,Jordy de Wijs,30,Left Center Back,True
9,Fortuna Düsseldorf,Dawid Kownacki,9,Center Attacking Midfield,True


### 1.3 Tracking-Daten

25 Bilder pro Sekunde mit den Positionen aller Spieler und des Balls. Das Hauptmodell nutzt nur die Event-Daten; das Tracking kommt erst in der Erweiterung (Notebook 04) zum Einsatz. Hier nur die ersten 10 Bilder (das Einlesen dauert trotzdem ca. 30 Sekunden, weil die Datei groß ist).

In [6]:
tracking = sportec.load_open_tracking_data(match_id=BEISPIEL, coordinates="secondspectrum", limit=10, only_alive=True)
tr = tracking.to_df()
print(f"{tracking.metadata.frame_rate} Bilder pro Sekunde, {tr.shape[1]} Spalten "
      f"(Ball + je 4 Spalten pro Spieler: x, y, zurückgelegte Distanz d, Geschwindigkeit s)")

# Ball und ein Spieler zur Ansicht, Spieler-ID durch den Namen ersetzt
spieler = next(p for p in heim.players if p.starting and f"{p.player_id}_x" in tr.columns)
tr[["frame_id", "timestamp", "ball_x", "ball_y", "ball_z", f"{spieler.player_id}_x", f"{spieler.player_id}_y"]] \
    .rename(columns={f"{spieler.player_id}_x": f"{spieler.name}_x", f"{spieler.player_id}_y": f"{spieler.name}_y"})

25 Bilder pro Sekunde, 97 Spalten (Ball + je 4 Spalten pro Spieler: x, y, zurückgelegte Distanz d, Geschwindigkeit s)


,frame_id,timestamp,ball_x,ball_y,ball_z,D. Ginczek_x,D. Ginczek_y
0,10000,0 days 00:00:00.000000,-5.67,0.11,0.16,-0.59,8.72
1,10001,0 days 00:00:00.040000,-6.28,0.08,0.16,-0.71,8.63
2,10002,0 days 00:00:00.080000,-6.85,0.05,0.19,-0.83,8.54
3,10003,0 days 00:00:00.120000,-7.43,0.04,0.19,-0.97,8.44
4,10004,0 days 00:00:00.160000,-7.99,0.02,0.19,-1.11,8.34
5,10005,0 days 00:00:00.200000,-8.57,-0.01,0.18,-1.25,8.24
6,10006,0 days 00:00:00.240000,-9.14,-0.03,0.15,-1.40,8.13
7,10007,0 days 00:00:00.280000,-9.67,-0.04,0.16,-1.55,8.03
8,10008,0 days 00:00:00.320000,-10.18,-0.06,0.18,-1.71,7.92
9,10009,0 days 00:00:00.360000,-10.69,-0.10,0.20,-1.88,7.81


## 2. Pass-Tabelle aller 7 Spiele

`pass_tabelle()` in `passdaten.py` macht aus den Events eines Spiels eine Zeile pro Pass. Dabei werden die Koordinaten so gedreht, dass das passende Team immer nach rechts angreift (x = +52,5 ist das gegnerische Tor), und die DFL-Rohangaben übernommen.

In [7]:
datasets = {m: lade_events(m) for m in MATCH_IDS}                                # ca. 1 Minute
paesse = pd.concat([pass_tabelle(m, ds) for m, ds in datasets.items()], ignore_index=True)
print(f"{len(paesse)} Zuspiele, davon {paesse['erfolg'].mean():.1%} erfolgreich\n")
paesse[["match_id", "team", "player", "x", "y", "standard", "flanke", "hoehe", "play_angle", "bewertung", "erfolg"]].head(10)

6062 Zuspiele, davon 78.4% erfolgreich



,match_id,team,player,x,y,standard,flanke,hoehe,play_angle,bewertung,erfolg
0,J03WMX,FC Bayern München,Serge Gnabry,0.00,0.00,KICK_OFF,False,flat,179.19,successfullyCompleted,1
1,J03WMX,FC Bayern München,Dayot Upamecano,-14.23,-0.20,OPEN_PLAY,False,high,320.43,unsuccessful,0
2,J03WMX,1. FC Köln,B. Schmitz,-27.15,-34.00,THROW_IN,False,high,348.75,unsuccessful,0
3,J03WMX,FC Bayern München,Ryan Gravenberch,2.71,21.96,OPEN_PLAY,False,flat,118.17,successfullyCompleted,1
4,J03WMX,FC Bayern München,Leroy Sané,8.22,-17.85,OPEN_PLAY,False,flat,182.63,successfullyCompleted,1
5,J03WMX,FC Bayern München,Benjamin Pavard,-15.73,-16.75,OPEN_PLAY,False,flat,247.76,successfullyCompleted,1
6,J03WMX,FC Bayern München,Dayot Upamecano,-21.84,-1.81,OPEN_PLAY,False,flat,275.47,successfullyCompleted,1
7,J03WMX,FC Bayern München,M. de Ligt,-20.14,15.95,OPEN_PLAY,False,high,341.79,successfullyCompleted,1
8,J03WMX,FC Bayern München,Kingsley Coman,22.84,30.09,OPEN_PLAY,False,high,94.58,unsuccessful,0
9,J03WMX,FC Bayern München,Y. Sommer,-38.77,-0.70,OPEN_PLAY,False,flat,47.68,successfullyCompleted,1


In [8]:
# Übersicht je Spiel (Heim – Gast laut Metadaten)
spiele = paesse.groupby("match_id").agg(liga=("liga", "first"), zuspiele=("erfolg", "size"), erfolgsquote=("erfolg", "mean"))
spiele.insert(0, "spiel", {m: " – ".join(t.name for t in ds.metadata.teams) for m, ds in datasets.items()})
spiele.loc[MATCH_IDS].round(3)

,spiel,liga,zuspiele,erfolgsquote
match_id,,,,
J03WMX,1. FC Köln – FC Bayern München,1,984,0.813
J03WN1,VfL Bochum 1848 – Bayer 04 Leverkusen,1,728,0.720
J03WPY,Fortuna Düsseldorf – 1. FC Nürnberg,2,891,0.814
J03WOH,Fortuna Düsseldorf – SSV Jahn Regensburg,2,764,0.771
J03WQQ,Fortuna Düsseldorf – FC St. Pauli,2,925,0.804
J03WOY,Fortuna Düsseldorf – F.C. Hansa Rostock,2,867,0.742
J03WR9,Fortuna Düsseldorf – 1. FC Kaiserslautern,2,903,0.807


7 Spiele, rund 870 Zuspiele pro Spiel, knapp 80 % kommen an. Fortuna Düsseldorf ist in 5 der 7 Spiele dabei.

## 3. Was zählt als Pass?

Die DFL erfasst alle Zuspiele als `Play`, unterteilt in `Pass` und `Cross` (Flanke) und eingebettet in die Spielsituation (Standards).

In [9]:
paesse.groupby(["standard", "flanke"])["erfolg"].agg(Anzahl="count", Erfolgsquote="mean").round(3)

Anzahl  Erfolgsquote
standard    flanke                      
CORNER_KICK False        4         1.000
            True        53         0.396
FREE_KICK   False      156         0.737
            True        12         0.167
GOAL_KICK   False      119         0.706
KICK_OFF    False       32         1.000
OPEN_PLAY   False     5241         0.803
            True       140         0.293
THROW_IN    False      281         0.851
            True        24         0.292

Überlegungen:
- **Anstöße:** trivial, kommen immer an → raus.
- **Einwürfe:** mit der Hand, andere Technik und Situation → raus.
- **Ecken:** fast immer Flanken in einen dicht besetzten Strafraum, eher Torchance als Zuspiel → raus.
- **Freistöße und Abstöße:** echte Zuspiele mit dem Fuß, nur ohne Gegnerdruck → bleiben, mit eigenem Kennzeichen.
- **Flanken aus dem Spiel:** echte Zuspielversuche → bleiben, mit eigenem Kennzeichen `flanke`.

Ob die Kennzeichen nötig sind, prüft Notebook 02 (Abschnitt 2).

## 4. Wann ist ein Pass erfolgreich?

Die DFL bewertet jeden Pass (`Evaluation`): `successfullyCompleted` (angekommen), `successful` (Empfänger wurde gefoult) oder `unsuccessful` (nicht angekommen). Die ersten beiden zählen wir als angekommen.

Zum Vergleich hat kloppy ein eigenes Ergebnis (`COMPLETE` = angekommen). Kommen beide zum selben Urteil?

In [10]:
laut_dfl = paesse["erfolg"].map({1: "angekommen", 0: "nicht angekommen"})
laut_kloppy = (paesse["kloppy_ergebnis"] == "COMPLETE").map({True: "angekommen", False: "nicht angekommen"})
print(f"Gleiches Urteil bei {(laut_dfl == laut_kloppy).mean():.1%} der Pässe")
pd.crosstab(laut_dfl.rename("laut DFL"), laut_kloppy.rename("laut kloppy"))

Gleiches Urteil bei 99.9% der Pässe


laut kloppy,angekommen,nicht angekommen
laut DFL,,
angekommen,4750,4
nicht angekommen,0,1308


Lesart: In den Zeilen steht das Urteil der DFL, in den Spalten das von kloppy. Fast alle Pässe liegen auf der Diagonalen (beide sagen dasselbe). Nur 4 Pässe sind laut DFL angekommen, laut kloppy nicht (kloppy führt sie als „ins Aus“).

**Entscheidung:** Erfolg = offizielle DFL-Bewertung `successfullyCompleted` oder `successful`. Pässe ins Aus und ins Abseits zählen als Fehlpass.

## 5. Empfänger bei Fehlpässen

Wer steht bei Fehlpässen im Feld `Recipient`?

In [11]:
paesse["recipient_typ"] = np.where(
    paesse["recipient"].isna(), "keiner",
    np.where(paesse["recipient_team"] == paesse["team_id"], "Mitspieler", "Gegenspieler"),
)
pd.crosstab(paesse["bewertung"], paesse["recipient_typ"])

recipient_typ,Gegenspieler,Mitspieler,keiner
bewertung,,,
successful,0,0,23
successfullyCompleted,0,4731,0
unsuccessful,1172,0,136


Bei Fehlpässen ist `Recipient` der Gegenspieler, der den Ball bekommt, nie der gemeinte Mitspieler. Auch einen Endpunkt gibt es bei Fehlpässen nicht. **Wen der Passgeber anspielen wollte, fehlt also in den Daten.** Das Modell nutzt deshalb die *Richtung* des Passes, nicht sein Ziel. (Die Erweiterung in Notebook 04 schätzt den Empfänger aus den Tracking-Daten.)

## 6. Leakage: Welche Angaben dürfen ins Modell?

**Grundsatz:** Ins Modell darf nur, was im Moment des Abspiels feststeht. Alles, was das *Ende* des Passes beschreibt, kennt man erst hinterher und würde das Ergebnis verraten.

Das gilt sicher für Endpunkt und Empfänger. Bei den DFL-Zusatzangaben kommt hinzu: Sie werden von Datenerfassern eingetragen, die das Ergebnis schon kennen. Ein Beispiel ist die Passart (`Direction`):

In [12]:
print("Passart ausgefüllt, nach Ergebnis:")
print(paesse.groupby("erfolg")["direction_dfl"].apply(lambda s: s.notna().mean()).round(3).rename({0: "Fehlpass", 1: "angekommen"}))
paesse.groupby(paesse["direction_dfl"].fillna("(leer)"))["erfolg"].agg(Anzahl="count", Erfolgsquote="mean").round(3)

Passart ausgefüllt, nach Ergebnis:
erfolg
Fehlpass      0.296
angekommen    0.076
Name: direction_dfl, dtype: float64


,Anzahl,Erfolgsquote
direction_dfl,,
(leer),5312,0.827
backPass,49,0.939
diagonalBall,328,0.521
squarePass,64,0.844
throughBall,309,0.298


Bei Fehlpässen ist die Passart viermal so oft ausgefüllt wie bei angekommenen Pässen (30 % gegenüber 8 %). Allein „Passart ist ausgefüllt“ verrät also schon viel über das Ergebnis.

| Angabe | Entscheidung | Begründung |
|---|---|---|
| Abspielort, Standard, Flanke | verwenden | stehen beim Abspiel fest |
| `PlayAngle` (Richtung) | verwenden | steht beim Abspiel fest; nur bei abgefälschten Bällen könnte die DFL eher die Richtung zum Abfangpunkt eintragen (Notebook 04 nimmt deshalb die Richtung aus dem Ballweg) |
| `Height` (hoch/flach) | verwenden | ist beim Abspiel erkennbar |
| `Distance` (Länge) | nicht verwenden | beschreibt das Ende des Passes; bei Fehlpässen unklar, ob die geplante Länge oder die Strecke bis zum Abfangen gemeint ist |
| `Direction` (Passart) | nicht verwenden | wird vor allem bei Fehlpässen eingetragen (siehe oben) |
| Endpunkt, `Recipient` | nicht verwenden | gibt es nur bzw. anders bei Fehlpässen |

In [13]:
(ROOT / "data").mkdir(exist_ok=True)
paesse.to_csv(ROOT / "data" / "paesse.csv", index=False)
print("gespeichert: data/paesse.csv")

gespeichert: data/paesse.csv


## Ergebnis

| | Entscheidung |
|---|---|
| Pass | alle Zuspiele mit dem Fuß, ohne Anstöße, Einwürfe und Ecken; Freistöße, Abstöße und Flanken bleiben mit eigenem Kennzeichen |
| Erfolg | DFL-Bewertung `successfullyCompleted` oder `successful` |
| Empfänger | bei Fehlpässen unbekannt, deshalb nutzt das Modell die Richtung des Passes, nicht sein Ziel |
| Merkmale | nur Angaben, die beim Abspiel feststehen: Ort, Situation, Flanke, Richtung (`PlayAngle`), Höhe (`Height`) |

Für das Modell bleiben 5.664 Pässe (ohne Anstöße, Einwürfe, Ecken und 4 Pässe ohne Richtungsangabe; `pass_auswahl()` in `modell.py`).

**Weiter in Notebook 02:** das Passmodell auf dieser Grundlage.